In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# create tools for subagents to use. 
from langchain.tools import tool

@tool
def sq_root(x: float) -> float: 
    """calculate the square root of a number"""
    return x ** 0.5

@tool
def sq(x: float) -> float:
    """calculate the square of a number"""
    return x ** 2

In [ ]:
# create subagents
from langchain.agents import create_agent

subagent_1 = create_agent(
    model="claude-sonnet-4-5-20250929",
    tools=[sq_root]
)

subagent_2 = create_agent(
    model="claude-sonnet-4-5-20250929",
    tools=[sq]
)

In [4]:
# create tools for main agent to use
@tool
def call_subagent1(x: float) -> float: 
    """call subagent_1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [{"role": "user", "content": f"calculate the square root of {x}"}]})
    return response["messages"][-1].content

def call_subagent2(x: float) -> float:
    """call subagent_2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [{"role": "user", "content": f"calculate the square of {x}"}]})
    return response["messages"][-1].content

In [5]:
# create the main agent
main_agent = create_agent(
    model="claude-sonnet-4-5-20250929",
    tools=[call_subagent1, call_subagent2],
    system_prompt="You are an assistant that uses subagents to perform math functions."
)

In [6]:
# invoke the main agent
response = main_agent.invoke(
    {"messages": [{"role": "user", "content": "what is the square root of 916652"}]}
)

print(response)

{'messages': [HumanMessage(content='what is the square root of 916652', additional_kwargs={}, response_metadata={}, id='d302c9bc-1a2e-4d22-9b5c-1f391092090c'), AIMessage(content=[{'id': 'toolu_01Aefvbn9vQ3BwVC1L75b2xx', 'input': {'x': 916652}, 'name': 'call_subagent1', 'type': 'tool_use', 'caller': {'type': 'direct'}}], additional_kwargs={}, response_metadata={'id': 'msg_01Fv3nG9MZANZd7jxDpHKMBS', 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 662, 'output_tokens': 57, 'server_tool_use': None, 'service_tier': 'standard', 'inference_geo': 'not_available'}, 'model_name': 'claude-sonnet-4-5-20250929', 'model_provider': 'anthropic'}, id='lc_run--019c6181-9cd6-7692-adcc-47b61199d997-0', tool_calls=[{'name': 'call_subagent1', 'args': {'x': 916652}, 'id': 'toolu_01Aefvbn9vQ3BwVC1L

In [7]:
print(response["messages"][-1])

content='The square root of 916652 is approximately **957.42**.' additional_kwargs={} response_metadata={'id': 'msg_017nmw2MGMfBpEErzqdUKgyf', 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 749, 'output_tokens': 18, 'server_tool_use': None, 'service_tier': 'standard', 'inference_geo': 'not_available'}, 'model_name': 'claude-sonnet-4-5-20250929', 'model_provider': 'anthropic'} id='lc_run--019c6181-b245-7913-9c50-912db9b61156-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 749, 'output_tokens': 18, 'total_tokens': 767, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}}
